<div>
<center><img src="../assets/Flux-logo.svg" width="400"/>
</div>

# Chapter 2: Python Submission API 🐍️

Flux also provides first-class python bindings which can be used to submit jobs programmatically. 

### Importing the flux package

Flux requires python to build, so if you have Flux installed, you have at least one python installation that works. However, you can also `pip install flux-python` to get the `flux` package in a side-installation of python (which you may have to do in this notebook!)

In [10]:
!python3 -c "import flux; print(flux.Flux())"

In [2]:
import os
import json
import flux
from flux.job import JobspecV1
from flux.job.JobID import JobID

### `flux.job.JobspecV1` to create job specifications

Flux represents work as a standard called the [Jobspec](https://flux-framework.readthedocs.io/projects/flux-rfc/en/latest/spec_25.html). While you could write YAML or JSON, it's much easier to use provided Python functions that take high level metadata (command, resources, etc) to generate them. We can then replicate our previous example of submitting multiple heterogeneous jobs using these Python helpers, and testing that Flux co-schedules them.

In [35]:
# connect to the running Flux instance
f = flux.Flux()

# Create the Jobspec from a command to run a python script, and specify resources
for i in range(16):
    compute_jobreq = JobspecV1.from_command(
        command=["../ch1/hello"], num_tasks=i+1, num_nodes=1, cores_per_task=1
        )
    flux.job.submit(f, compute_jobreq)

# This is the "current working directory" (cwd)
#compute_jobreq.cwd = os.getcwd()

# When we submit, we get back the job identifier (JobID)
#for i in range(100):
#    flux.job.submit(f, compute_jobreq) # submit and print out the jobid (in f58 format)

Once we create the job, when we submit it in Python we get back a job identifier or jobid. We can then interact with the Flux handle, a connection to Flux, to get information about that job.

### `flux.job.get_job(handle, jobid)` to get job info

In [19]:
# Let's submit again to retrieve (and save) the job identifier
fluxjob = flux.job.submit(f, compute_jobreq)
fluxjobid = JobID(fluxjob.f58)
print(f"🎉️ Hooray, we just submitted {fluxjobid}!\n")

# Here is how to get your info. The first argument is the flux handle, then the jobid
jobinfo = flux.job.get_job(f, fluxjobid)
print(jobinfo)

🎉️ Hooray, we just submitted ƒ3Xve8ytQF!

{'t_depend': 1747697972.2159176, 't_run': 0.0, 't_cleanup': 0.0, 't_inactive': 0.0, 'duration': 0.0, 'expiration': 0.0, 'name': 'compute.py', 'cwd': '/home/hobbs17/flux-tutorials/tutorial/flux-for-lc/ch2', 'queue': '', 'project': '', 'bank': '', 'ntasks': 1, 'ncores': 1, 'nnodes': 1, 'priority': 16, 'ranks': '', 'nodelist': '', 'success': '', 'result': '', 'waitstatus': '', 'id': JobID(324407704682496), 't_submit': 1747697972.203312, 't_remaining': 0.0, 'state': 'SCHED', 'username': 'hobbs17', 'userid': 60943, 'urgency': 16, 'runtime': 0.0, 'status': 'SCHED', 'returncode': '', 'dependencies': [], 'annotations': {}, 'exception': {'occurred': '', 'severity': '', 'type': '', 'note': ''}}


You can now run `flux jobs` to see the jobs that we submit from Python.

In [21]:
!flux jobs -a --name="compute.py"

       JOBID USER     NAME       ST NTASKS NNODES     TIME INFO
  ƒ3Xve8ytQF hobbs17  compute.py CD      1      1   3.057s auk108.llnl.gov
  ƒ3XbeUmZnX hobbs17  compute.py CD      1      1   3.064s auk108.llnl.gov
  ƒ3XZ6ghANj hobbs17  compute.py CD      1      1   3.049s auk108.llnl.gov
  ƒ3XJK5pcTR hobbs17  compute.py CD      1      1   3.053s auk108.llnl.gov
  ƒ3XDstTCum hobbs17  compute.py CD      1      1   3.053s auk108.llnl.gov
  ƒ3XDsGsW9V hobbs17  compute.py CD      1      1   3.047s auk108.llnl.gov
  ƒ3XDrdop6s hobbs17  compute.py CD      1      1   3.051s auk108.llnl.gov
  ƒ3XDqzk84F hobbs17  compute.py CD      1      1   3.053s auk108.llnl.gov
  ƒ3XDqPARHy hobbs17  compute.py CD      1      1   3.031s auk108.llnl.gov
  ƒ3XDpk6jFM hobbs17  compute.py CD      1      1   3.029s auk108.llnl.gov
  ƒ3XDp733Cj hobbs17  compute.py CD      1      1   3.036s auk108.llnl.gov
  ƒ3XDoVTLST hobbs17  compute.py CD      1      1   3.079s auk108.llnl.gov
  ƒ3XDkuiayd hobbs17  compute.py CD 

Under the hood, the `Jobspec` class is creating a YAML document that ultimately gets serialized as JSON and sent to Flux for ingestion, validation, queueing, scheduling, and eventually execution.  We can dump the raw JSON jobspec that is submitted, where we can see the exact resources requested and the task set to be executed on those resources.

In [49]:
print(compute_jobreq.dumps(indent=2))

{
  "resources": [
    {
      "type": "node",
      "count": 1,
      "with": [
        {
          "type": "slot",
          "count": 1,
          "with": [
            {
              "type": "core",
              "count": 1
            }
          ],
          "label": "task"
        }
      ]
    }
  ],
  "tasks": [
    {
      "command": [
        "./compute.py",
        "120"
      ],
      "slot": "task",
      "count": {
        "per_slot": 1
      }
    }
  ],
  "attributes": {
    "system": {
      "duration": 0,
      "cwd": "/home/jovyan/flux-tutorial/flux-workflow-examples/job-submit-api/"
    }
  },
  "version": 1
}


### Playing with the Synchronous job submission API



### `FluxExecutor` for bulk submission

We can use the FluxExecutor class to submit large numbers of jobs to Flux. This method resembles python's `concurrent.futures` interface.

In [30]:
from flux.job import FluxExecutor
with FluxExecutor() as executor:
    compute_jobspec = JobspecV1.from_command(["sleep", "3"])
    futures = [executor.submit(compute_jobspec) for _ in range(10)]
    # wait for the jobid for each job, as a proxy for the job being submitted
    print(futures)
    # all jobs submitted - print timings
print("I'm all done")

[<FluxExecutorFuture at 0x7f689831ab50 state=running>, <FluxExecutorFuture at 0x7f687e1d9e20 state=running>, <FluxExecutorFuture at 0x7f687e1d9ac0 state=running>, <FluxExecutorFuture at 0x7f687f314460 state=running>, <FluxExecutorFuture at 0x7f687f314e80 state=running>, <FluxExecutorFuture at 0x7f687f314ca0 state=running>, <FluxExecutorFuture at 0x7f687f314ee0 state=running>, <FluxExecutorFuture at 0x7f687f3141c0 state=pending>, <FluxExecutorFuture at 0x7f687f3143d0 state=pending>, <FluxExecutorFuture at 0x7f687f3eb340 state=pending>]
I'm all done


In [31]:
# Submit the FluxExecutor based script.
!flux python bulksubmit_executor.py -n200 /bin/sleep 0

bulksubmit_executor: submitted 200 jobs in 0.11s. 1785.83job/s
bulksubmit_executor: First job finished in about 0.177s
|██████████████████████████████████████████████████████████| 100.0% (286.2 job/s)
bulksubmit_executor: Ran 200 jobs in 0.9s. 221.7 job/s


In [ ]:
import concurrent.futures
import flux.job

jobspec = flux.job.JobspecV1.from_command(["/bin/true"])
with flux.job.FluxExecutor() as executor:
        futs = [executor.submit(jobspec) for _ in range(5)]
        for f in concurrent.futures.as_completed(futs):
                print(f.result())

### `flux.event_watch` to watch events

If you want to get the output of a job (or more generally, stream events) you can do that as follows. Let's submit a quick job, and then look at the output.


In [53]:
# Create the Jobspec from a command to run a python script, and specify resources
jobspec = JobspecV1.from_command(
    command=["echo", "Flux Plumbing 💩️🚽️"], num_tasks=1, num_nodes=1, cores_per_task=1)
jobid = flux.job.submit(f, jobspec)

# Give some time to run and finish
import time
time.sleep(5)

for line in flux.job.event_watch(f, jobid, "guest.output"):
    print(line)

1721520256.00717: header {'version': 1, 'encoding': {'stdout': 'UTF-8', 'stderr': 'UTF-8'}, 'count': {'stdout': 1, 'stderr': 1}, 'options': {}}
1721520256.01083: data {'stream': 'stderr', 'rank': '0', 'eof': True}
1721520256.01085: data {'stream': 'stdout', 'rank': '0', 'data': 'Flux Plumbing 💩️🚽️\n'}
1721520256.01087: data {'stream': 'stdout', 'rank': '0', 'eof': True}


### `flux.job.job_list` to list jobs

Finally, it can be really helpful to get an entire listing of jobs. You can do that as follows. Note that the `job_list` is creating a remote procedure call (rpc) and we call `get` to retrieve the output.

In [ ]:
flux.job.job_list(f).get()